# Syria Population Raster Aggregation (Approximately 1 km)

Overview

Aggregates the WorldPop 2026 population raster for Syria from its source resolution of approximately 100 m to an output resolution of approximately 1 km.
The source values represent estimated population per source grid cell. The output raster is therefore created with sum resampling so that each output cell contains the sum of the contributing source cells.
The workflow validates the source metadata and values, performs the aggregation, compares population totals before and after processing, saves the result as a GeoTIFF, and creates an interactive map.

WorldPop 2026のシリア人口ラスタを、約100mの元解像度から約1kmの出力解像度へ集約します。
元データの値は、元グリッドセルごとの推計人口を表しています。そのため、出力セルには対応する元セルの合計人口が格納されるよう、合計によるリサンプリングを使用します。
元Rasterのメタデータと値を検証し、集約前後の人口合計を比較します。処理結果をGeoTIFFとして保存し、インタラクティブ地図を作成します。

Objectives

- Read and inspect the WorldPop 2026 population raster
- Validate the CRS, dimensions, transform, NoData value and data type
- Aggregate the raster to one tenth of its source width and height
- Use sum resampling to preserve estimated population totals
- Compare valid pixels and population totals before and after aggregation
- Save the aggregated raster as a GeoTIFF
- Read back and validate the saved raster
- Create an interactive map with administrative boundaries, a legend and an information panel

- WorldPop 2026人口ラスタを読み込み、内容を確認する
- CRS、サイズ、Transform、NoData値およびデータ型を検証する
- 元Rasterの幅と高さを10分の1へ集約する
- 推計人口合計を保持するため、合計によるリサンプリングを使用する
- 集約前後の有効セル数と人口合計を比較する
- 集約結果をGeoTIFFとして保存する
- 保存したRasterを再読込して検証する
- 行政界、凡例および情報パネルを備えたインタラクティブ地図を作成する

Workflow

#### English

1. Define the source and output paths
2. Read and validate the source raster metadata
3. Read the source values as a masked array
4. Calculate source raster statistics
5. Define the output dimensions and transform
6. Aggregate source cells using sum resampling
7. Compare source and aggregated population totals
8. Save the aggregated raster as a GeoTIFF
9. Read back and validate the saved raster
10. Prepare a display array and create an interactive map

#### 日本語

1. 入力データと出力先のパスを定義する
2. 元Rasterのメタデータを読み込み、検証する
3. 元RasterをMaskedArrayとして読み込む
4. 元Rasterの統計値を計算する
5. 出力サイズとTransformを定義する
6. 合計によるリサンプリングで元セルを集約する
7. 元Rasterと集約後Rasterの人口合計を比較する
8. 集約結果をGeoTIFFとして保存する
9. 保存したRasterを再読込して検証する
10. 表示用配列を作成し、インタラクティブ地図を作成する

Data

Population raster data:

- `worldpop_syria_2026.tif`

Source: WorldPop, open population data

Administrative boundary data:

- `syr_admin0.geojson`
- `syr_admin1.geojson`

Source: HDX OCHA, Syria subnational administrative boundaries

Technologies

- Python
- Rasterio
- NumPy
- GeoPandas
- Folium
- Matplotlib
- GeoTIFF

In [ ]:
# 1
# Import the required libraries
# 必要なライブラリを読み込む

# File paths
# ファイルパス
from pathlib import Path

# Numerical processing
# 数値処理
import numpy as np

# Raster processing
# Raster処理
import rasterio
from affine import Affine
from rasterio.enums import Resampling
from rasterio.transform import array_bounds
from rasterio.warp import reproject

# Vector processing
# Vector処理
import geopandas as gpd

# Web mapping and colour definition
# Web地図作成と色の定義
import folium
import matplotlib.colors as mcolors
from branca.element import Element

In [ ]:
# 2
# Define the source and output paths
# 入力データと出力先のパスを定義する

PROJECT_DIR = Path(
    "/Users/marisa/Syria_Humanitarian_Climate_Facts/" "01_PROJECTS/04_RASTER_OPERATIONS"
)

RASTER_DATA_DIR = Path(
    "/Users/marisa/Syria_Humanitarian_Climate_Facts/" "02_DATA/RASTER"
)

VECTOR_DATA_DIR = Path(
    "/Users/marisa/Syria_Humanitarian_Climate_Facts/" "02_DATA/VECTOR"
)

OUTPUT_DIR = PROJECT_DIR / "outputs"

population_raster_path = RASTER_DATA_DIR / "worldpop_syria_2026.tif"

admin0_path = VECTOR_DATA_DIR / "syr_admin0.geojson"

admin1_path = VECTOR_DATA_DIR / "syr_admin1.geojson"

aggregated_raster_path = OUTPUT_DIR / "worldpop_syria_2026_aggregated_approx_1km.tif"

output_path = PROJECT_DIR / "01_syria_raster_resample.html"

required_input_paths = {
    "Population raster": population_raster_path,
    "Country boundary": admin0_path,
    "Governorate boundaries": admin1_path,
}

missing_input_paths = [
    path for path in required_input_paths.values() if not path.exists()
]

if missing_input_paths:
    raise FileNotFoundError(
        "One or more required input files were not found:\n"
        + "\n".join(str(path) for path in missing_input_paths)
    )

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

print(f"Project directory: {PROJECT_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Source raster: {population_raster_path}")

In [ ]:
# 3
# Read the source raster metadata
# 元Rasterのメタデータを読み込む

with rasterio.open(population_raster_path) as src:

    source_crs = src.crs
    source_width = src.width
    source_height = src.height
    source_count = src.count
    source_transform = src.transform
    source_bounds = src.bounds
    source_nodata = src.nodata
    source_dtype = src.dtypes[0]
    source_profile = src.profile.copy()
    source_resolution = src.res

print(f"Source CRS: {source_crs}")
print("Source dimensions: " f"{source_width:,} × {source_height:,}")
print(f"Source band count: {source_count}")
print(f"Source data type: {source_dtype}")
print(f"Source NoData: {source_nodata}")
print(f"Source resolution: {source_resolution}")
print(f"Source bounds: {source_bounds}")
print(f"Source transform:\n{source_transform}")

In [ ]:
# 4
# Validate the source raster metadata
# 元Rasterのメタデータを検証する

if source_crs is None:
    raise ValueError("The source population raster has no defined CRS.")

if source_crs.to_epsg() != 4326:
    raise ValueError(
        "The source population raster was expected to use "
        f"EPSG:4326, but its CRS is {source_crs}."
    )

if source_count != 1:
    raise ValueError(
        "The source population raster was expected to "
        f"contain one band, but it contains {source_count}."
    )

if source_width <= 0 or source_height <= 0:
    raise ValueError("The source raster dimensions are not valid.")

if source_nodata is None:
    raise ValueError("The source population raster has no defined " "NoData value.")

if source_transform.a <= 0:
    raise ValueError("The source raster has an invalid horizontal " "pixel size.")

if source_transform.e >= 0:
    raise ValueError("The source raster was expected to be " "north-up.")

calculated_source_bounds = array_bounds(
    source_height,
    source_width,
    source_transform,
)

bounds_tolerance = 1e-9

if not np.allclose(
    calculated_source_bounds,
    tuple(source_bounds),
    atol=bounds_tolerance,
    rtol=0.0,
):
    raise ValueError(
        "The bounds calculated from the source transform "
        "do not match the stored raster bounds."
    )

print("Source raster metadata validation: passed")

print(
    "Calculated source bounds:",
    calculated_source_bounds,
)

In [ ]:
# 5
# Read and validate the source population values
# 元Rasterの人口値を読み込み、検証する

with rasterio.open(population_raster_path) as src:

    source_population = src.read(
        1,
        masked=True,
    )

if not np.ma.isMaskedArray(source_population):
    raise TypeError("The source population raster was not read " "as a masked array.")

source_valid_pixel_count = int(source_population.count())

source_masked_pixel_count = int(source_population.size - source_valid_pixel_count)

if source_valid_pixel_count == 0:
    raise ValueError("The source population raster contains no " "valid pixels.")

source_min = float(source_population.min())

source_max = float(source_population.max())

source_mean = float(source_population.mean())

source_population_total = float(source_population.sum(dtype=np.float64))

if source_min < 0:
    raise ValueError("The valid source population values contain " "negative values.")

if not np.isfinite(source_population_total):
    raise ValueError("The source population total is not finite.")

print(f"Source valid pixels: " f"{source_valid_pixel_count:,}")

print(f"Source masked pixels: " f"{source_masked_pixel_count:,}")

print(f"Source minimum value: {source_min:,.2f}")

print(f"Source maximum value: {source_max:,.2f}")

print(f"Source mean value: {source_mean:,.2f}")

print("Source estimated population total: " f"{source_population_total:,.0f}")

In [ ]:
# 6
# Define the aggregation dimensions and output transform
# 集約後のサイズと出力Transformを定義する

AGGREGATION_FACTOR = 10

output_width = int(np.ceil(source_width / AGGREGATION_FACTOR))

output_height = int(np.ceil(source_height / AGGREGATION_FACTOR))

width_scale = source_width / output_width

height_scale = source_height / output_height

output_transform = source_transform * Affine.scale(
    width_scale,
    height_scale,
)

output_resolution = (
    abs(output_transform.a),
    abs(output_transform.e),
)

output_bounds = array_bounds(
    output_height,
    output_width,
    output_transform,
)

if output_width >= source_width:
    raise ValueError("The output width was not reduced.")

if output_height >= source_height:
    raise ValueError("The output height was not reduced.")

if not np.allclose(
    output_bounds,
    tuple(source_bounds),
    atol=bounds_tolerance,
    rtol=0.0,
):
    raise ValueError(
        "The output transform does not retain the " "source raster bounds."
    )

print("Output dimensions: " f"{output_width:,} × {output_height:,}")

print(f"Width reduction ratio: {width_scale:.4f}")

print(f"Height reduction ratio: {height_scale:.4f}")

print(f"Output resolution: {output_resolution}")

print(f"Output bounds: {output_bounds}")

print(f"Output transform:\n{output_transform}")

In [ ]:
# 7
# Aggregate the population raster using sum resampling
# 合計によるリサンプリングで人口Rasterを集約する

aggregated_output_data = np.full(
    (
        output_height,
        output_width,
    ),
    source_nodata,
    dtype=np.float64,
)

with rasterio.open(population_raster_path) as src:

    reproject(
        source=rasterio.band(
            src,
            1,
        ),
        destination=aggregated_output_data,
        src_transform=source_transform,
        src_crs=source_crs,
        src_nodata=source_nodata,
        dst_transform=output_transform,
        dst_crs=source_crs,
        dst_nodata=source_nodata,
        resampling=Resampling.sum,
        init_dest_nodata=True,
    )

aggregated_population = np.ma.masked_equal(
    aggregated_output_data,
    source_nodata,
)

if aggregated_population.shape != (
    output_height,
    output_width,
):
    raise ValueError(
        "The aggregated raster dimensions do not match "
        "the defined output dimensions."
    )

if not np.ma.isMaskedArray(aggregated_population):
    raise TypeError(
        "The aggregated population raster was not " "created as a masked array."
    )

aggregated_valid_pixel_count = int(aggregated_population.count())

if aggregated_valid_pixel_count == 0:
    raise ValueError("The aggregated population raster contains " "no valid pixels.")

aggregated_min = float(aggregated_population.min())

aggregated_max = float(aggregated_population.max())

aggregated_mean = float(aggregated_population.mean())

aggregated_population_total = float(aggregated_population.sum(dtype=np.float64))

if aggregated_min < 0:
    raise ValueError(
        "The valid aggregated population values " "contain negative values."
    )

if not np.isfinite(aggregated_population_total):
    raise ValueError("The aggregated population total is not finite.")

print("Aggregation method: " "Rasterio warp with Resampling.sum")

print(f"Aggregated raster shape: " f"{aggregated_population.shape}")

print("Aggregated valid pixels: " f"{aggregated_valid_pixel_count:,}")

print(f"Aggregated minimum value: " f"{aggregated_min:,.2f}")

print(f"Aggregated maximum value: " f"{aggregated_max:,.2f}")

print(f"Aggregated mean value: " f"{aggregated_mean:,.2f}")

print("Aggregated estimated population total: " f"{aggregated_population_total:,.2f}")

In [ ]:
# 8
# Compare population totals before and after aggregation
# 集約前後の人口合計を比較する

if source_population_total <= 0:
    raise ValueError("The source population total must be greater " "than zero.")

population_difference = aggregated_population_total - source_population_total

absolute_population_difference = abs(population_difference)

population_difference_percent = (
    absolute_population_difference / source_population_total * 100
)

population_preservation_percent = (
    aggregated_population_total / source_population_total * 100
)

MAX_POPULATION_DIFFERENCE_PERCENT = 0.1

if population_difference_percent > MAX_POPULATION_DIFFERENCE_PERCENT:
    raise ValueError(
        "The population difference after aggregation "
        "exceeds the permitted threshold. "
        f"Difference: {population_difference_percent:.6f}%"
    )

print("Source estimated population total: " f"{source_population_total:,.2f}")

print("Aggregated estimated population total: " f"{aggregated_population_total:,.2f}")

print("Population difference: " f"{population_difference:,.2f}")

print("Absolute difference percentage: " f"{population_difference_percent:.6f}%")

print("Population preservation percentage: " f"{population_preservation_percent:.6f}%")

In [ ]:
# 9
# Prepare the output raster array and metadata
# 出力Rasterの配列とメタデータを準備する

OUTPUT_DTYPE = "float32"

aggregated_output_array = aggregated_population.filled(source_nodata).astype(
    OUTPUT_DTYPE
)

output_profile = source_profile.copy()

output_profile.update(
    driver="GTiff",
    count=1,
    width=output_width,
    height=output_height,
    crs=source_crs,
    transform=output_transform,
    dtype=OUTPUT_DTYPE,
    nodata=source_nodata,
    compress="LZW",
    predictor=3,
    tiled=True,
)

if aggregated_output_array.shape != (
    output_height,
    output_width,
):
    raise ValueError("The output array dimensions do not match " "the output metadata.")

if aggregated_output_array.dtype != np.dtype(OUTPUT_DTYPE):
    raise TypeError("The output array does not use the defined " "data type.")

print(f"Output array shape: " f"{aggregated_output_array.shape}")

print(f"Output array data type: " f"{aggregated_output_array.dtype}")

print(f"Output NoData value: {source_nodata}")

print("Output compression: " f"{output_profile['compress']}")

In [ ]:
# 10
# Save the aggregated population raster as a GeoTIFF
# 集約した人口RasterをGeoTIFFとして保存する

with rasterio.open(
    aggregated_raster_path,
    "w",
    **output_profile,
) as dst:

    dst.write(
        aggregated_output_array,
        1,
    )

print(f"Aggregated raster saved to: " f"{aggregated_raster_path}")

print("Output file size: " f"{aggregated_raster_path.stat().st_size / 1024**2:,.2f} MB")

In [ ]:
# 11
# Read back the saved aggregated raster
# 保存した集約Rasterを再読込する

with rasterio.open(aggregated_raster_path) as src:

    saved_population = src.read(
        1,
        masked=True,
    )

    saved_crs = src.crs
    saved_width = src.width
    saved_height = src.height
    saved_count = src.count
    saved_transform = src.transform
    saved_bounds = src.bounds
    saved_resolution = src.res
    saved_nodata = src.nodata
    saved_dtype = src.dtypes[0]
    saved_compression = src.compression

saved_valid_pixel_count = int(saved_population.count())

saved_population_total = float(saved_population.sum(dtype=np.float64))

print(f"Saved CRS: {saved_crs}")

print("Saved dimensions: " f"{saved_width:,} × {saved_height:,}")

print(f"Saved band count: {saved_count}")
print(f"Saved data type: {saved_dtype}")
print(f"Saved NoData: {saved_nodata}")
print(f"Saved resolution: {saved_resolution}")
print(f"Saved bounds: {saved_bounds}")
print(f"Saved compression: {saved_compression}")

print("Saved valid pixels: " f"{saved_valid_pixel_count:,}")

print("Saved estimated population total: " f"{saved_population_total:,.2f}")

In [ ]:
# 12
# Validate the saved aggregated raster
# 保存した集約Rasterを検証する

if saved_crs != source_crs:
    raise ValueError("The saved raster does not retain the source CRS.")

if saved_width != output_width:
    raise ValueError(
        "The saved raster width does not match the " "defined output width."
    )

if saved_height != output_height:
    raise ValueError(
        "The saved raster height does not match the " "defined output height."
    )

if saved_count != 1:
    raise ValueError("The saved raster does not contain one band.")

if saved_transform != output_transform:
    raise ValueError("The saved raster does not retain the output " "transform.")

if not np.allclose(
    tuple(saved_bounds),
    output_bounds,
    atol=bounds_tolerance,
    rtol=0.0,
):
    raise ValueError(
        "The saved raster bounds do not match the " "defined output bounds."
    )

if saved_nodata != source_nodata:
    raise ValueError("The saved raster does not retain the source " "NoData value.")

if saved_dtype != OUTPUT_DTYPE:
    raise ValueError("The saved raster does not use the defined " "output data type.")

if saved_population.shape != (
    output_height,
    output_width,
):
    raise ValueError("The saved raster array has unexpected " "dimensions.")

saved_total_difference = abs(saved_population_total - aggregated_population_total)

saved_total_difference_percent = (
    saved_total_difference / aggregated_population_total * 100
)

if saved_total_difference_percent > MAX_POPULATION_DIFFERENCE_PERCENT:
    raise ValueError(
        "The saved raster population total differs "
        "from the in-memory aggregated total."
    )

print("Saved raster validation: passed")

print("Saved-total difference percentage: " f"{saved_total_difference_percent:.6f}%")

In [ ]:
# 13
# Read and validate the administrative boundary datasets
# 行政界データを読み込み、検証する

admin0 = gpd.read_file(admin0_path)

admin1 = gpd.read_file(admin1_path)

administrative_datasets = {
    "Country boundaries": admin0,
    "Governorate boundaries": admin1,
}

for dataset_name, dataset in administrative_datasets.items():

    if dataset.crs is None:
        raise ValueError(f"{dataset_name} has no defined CRS.")

    if dataset.crs != saved_crs:
        raise ValueError(
            f"{dataset_name} does not use the same "
            "CRS as the saved population raster."
        )

    if dataset.empty:
        raise ValueError(f"{dataset_name} contains no features.")

    if dataset.geometry.isna().any():
        raise ValueError(f"{dataset_name} contains missing geometries.")

    if dataset.geometry.is_empty.any():
        raise ValueError(f"{dataset_name} contains empty geometries.")

    if not dataset.geometry.is_valid.all():
        raise ValueError(f"{dataset_name} contains invalid geometries.")

    print(f"{dataset_name}: " f"{len(dataset):,} features, {dataset.crs}")

In [ ]:
# 14
# Prepare the aggregated raster for web display
# 集約RasterをWeb表示用に準備する

saved_population_mask = np.ma.getmaskarray(saved_population)

saved_valid_values = saved_population.compressed()

if saved_valid_values.size == 0:
    raise ValueError("The saved population raster contains no " "values for display.")

DISPLAY_PERCENTILE = 99.0

display_upper_value = float(
    np.percentile(
        saved_valid_values,
        DISPLAY_PERCENTILE,
    )
)

if display_upper_value <= 0:
    raise ValueError("The display upper value must be greater " "than zero.")

clipped_population = np.clip(
    saved_population.filled(np.nan),
    0,
    display_upper_value,
)

display_population = np.full(
    saved_population.shape,
    np.nan,
    dtype=np.float32,
)

valid_display_pixels = ~saved_population_mask & np.isfinite(clipped_population)

display_population[valid_display_pixels] = np.log1p(
    clipped_population[valid_display_pixels]
) / np.log1p(display_upper_value)

west, south, east, north = array_bounds(
    saved_height,
    saved_width,
    saved_transform,
)

display_bounds = [
    [
        south,
        west,
    ],
    [
        north,
        east,
    ],
]

if not np.isfinite(display_population[valid_display_pixels]).all():
    raise ValueError(
        "The display population array contains " "non-finite valid values."
    )

print(f"Display percentile: " f"{DISPLAY_PERCENTILE:.1f}")

print("Display upper value: " f"{display_upper_value:,.2f}")

print(f"Display bounds: {display_bounds}")

print("Display valid pixels: " f"{np.count_nonzero(valid_display_pixels):,}")

In [ ]:
# 15
# Define the colour map and legend values
# カラーマップと凡例の値を定義する

colour_stops = [
    (
        0.00,
        "#7bb5a000",
    ),
    (
        0.20,
        "#7bb5a0ff",
    ),
    (
        0.40,
        "#2e9166ff",
    ),
    (
        0.60,
        "#226c4cff",
    ),
    (
        0.80,
        "#184d36ff",
    ),
    (
        1.00,
        "#0f3122ff",
    ),
]

population_cmap = mcolors.LinearSegmentedColormap.from_list(
    "SyriaPopulationAggregation",
    colour_stops,
)

population_cmap.set_bad(
    color=(
        0,
        0,
        0,
        0,
    )
)

legend_positions = np.array(
    [
        0.00,
        0.25,
        0.50,
        0.75,
        1.00,
    ]
)

legend_values = np.expm1(legend_positions * np.log1p(display_upper_value))

print(
    "Legend values:",
    [round(value) for value in legend_values],
)

In [ ]:
# 16
# Create the no-label base map and set the initial extent
# 地名表記のないベースマップを作成し、初期表示範囲を設定する

m = folium.Map(
    location=[
        34.8,
        38.5,
    ],
    zoom_start=6,
    tiles=None,
    control_scale=True,
    prefer_canvas=True,
)

folium.TileLayer(
    tiles=("https://{s}.basemaps.cartocdn.com/" "light_nolabels/{z}/{x}/{y}{r}.png"),
    attr=(
        "&copy; "
        '<a href="https://www.openstreetmap.org/copyright">'
        "OpenStreetMap</a> contributors "
        "&copy; "
        '<a href="https://carto.com/attributions">'
        "CARTO</a>"
    ),
    name="CARTO Light — No Labels",
    control=True,
    show=True,
).add_to(m)

m.fit_bounds(
    display_bounds,
    padding=[
        25,
        25,
    ],
)

In [ ]:
# 17
# Add the aggregated population raster layer
# 集約した人口Rasterレイヤーを追加する

population_raster_layer = folium.FeatureGroup(
    name="Aggregated Population (Approximately 1 km)",
    show=True,
)

folium.raster_layers.ImageOverlay(
    image=display_population,
    bounds=display_bounds,
    colormap=population_cmap,
    opacity=0.82,
    name="Aggregated Population",
    interactive=True,
    zindex=1,
).add_to(population_raster_layer)

population_raster_layer.add_to(m)

In [ ]:
# 18
# Add the country and governorate boundaries
# 国境および県境レイヤーを追加する

country_boundary_layer = folium.FeatureGroup(
    name="Country Boundary",
    show=True,
)

folium.GeoJson(
    data=admin0[
        [
            "adm0_name",
            "adm0_pcode",
            "geometry",
        ]
    ].to_json(),
    name="Country Boundary",
    style_function=lambda feature: {
        "fillColor": "transparent",
        "color": "#202020",
        "weight": 2.8,
        "fillOpacity": 0.0,
        "opacity": 0.95,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=[
            "adm0_name",
            "adm0_pcode",
        ],
        aliases=[
            "Country:",
            "Pcode:",
        ],
        sticky=False,
    ),
).add_to(country_boundary_layer)

country_boundary_layer.add_to(m)

governorate_boundary_layer = folium.FeatureGroup(
    name="Governorate Boundaries",
    show=True,
)

folium.GeoJson(
    data=admin1[
        [
            "adm1_name",
            "adm1_pcode",
            "geometry",
        ]
    ].to_json(),
    name="Governorate Boundaries",
    style_function=lambda feature: {
        "fillColor": "transparent",
        "color": "#666666",
        "weight": 1.0,
        "fillOpacity": 0.0,
        "opacity": 0.85,
    },
    highlight_function=lambda feature: {
        "color": "#111111",
        "weight": 2.0,
        "opacity": 1.0,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=[
            "adm1_name",
            "adm1_pcode",
        ],
        aliases=[
            "Governorate:",
            "Pcode:",
        ],
        sticky=False,
    ),
).add_to(governorate_boundary_layer)

governorate_boundary_layer.add_to(m)

In [ ]:
# 19
# Add governorate and neighbouring-country labels
# 県名および周辺国名を追加する

governorate_label_layer = folium.FeatureGroup(
    name="Governorate Labels",
    show=True,
)

admin1_label_points = admin1.copy()

if {
    "center_lat",
    "center_lon",
}.issubset(admin1_label_points.columns) and admin1_label_points[
    [
        "center_lat",
        "center_lon",
    ]
].notna().all().all():
    admin1_label_points["label_latitude"] = admin1_label_points["center_lat"]

    admin1_label_points["label_longitude"] = admin1_label_points["center_lon"]

else:
    representative_points = admin1_label_points.to_crs(
        "EPSG:32637"
    ).geometry.representative_point()

    representative_points = gpd.GeoSeries(
        representative_points,
        crs="EPSG:32637",
    ).to_crs(saved_crs)

    admin1_label_points["label_latitude"] = representative_points.y

    admin1_label_points["label_longitude"] = representative_points.x

for _, governorate in admin1_label_points.iterrows():

    folium.Marker(
        location=[
            governorate["label_latitude"],
            governorate["label_longitude"],
        ],
        icon=folium.DivIcon(
            icon_size=(
                130,
                24,
            ),
            icon_anchor=(
                65,
                12,
            ),
            html=f"""
            <div style="
                font-size: 12px;
                font-weight: 700;
                color: #202020;
                text-align: center;
                white-space: nowrap;
                text-shadow:
                    -1px -1px 0 #ffffff,
                     1px -1px 0 #ffffff,
                    -1px  1px 0 #ffffff,
                     1px  1px 0 #ffffff;
            ">
                {governorate["adm1_name"]}
            </div>
            """,
        ),
    ).add_to(governorate_label_layer)

governorate_label_layer.add_to(m)

neighbour_label_layer = folium.FeatureGroup(
    name="Neighbour Labels",
    show=True,
)

neighbour_labels = {
    "TÜRKİYE": [
        37.75,
        38.1,
    ],
    "IRAQ": [
        34.5,
        42.75,
    ],
    "LEBANON": [
        33.8,
        35.5,
    ],
    "JORDAN": [
        31.85,
        37.2,
    ],
}

for country_name, coordinates in neighbour_labels.items():

    folium.Marker(
        location=coordinates,
        icon=folium.DivIcon(
            icon_size=(
                150,
                30,
            ),
            icon_anchor=(
                75,
                15,
            ),
            html=f"""
            <div style="
                font-size: 18px;
                font-weight: 700;
                color: #666666;
                text-align: center;
                white-space: nowrap;
                text-shadow:
                    -1px -1px 0 #ffffff,
                     1px -1px 0 #ffffff,
                    -1px  1px 0 #ffffff,
                     1px  1px 0 #ffffff;
            ">
                {country_name}
            </div>
            """,
        ),
    ).add_to(neighbour_label_layer)

neighbour_label_layer.add_to(m)

In [ ]:
# 20
# Add the map information and source panel
# 地図の説明、処理方法および出典を追加する

information_panel_html = f"""
<div style="
    position: fixed;
    top: 20px;
    left: 50px;
    width: 440px;
    min-height: 230px;
    background-color: rgba(255, 255, 255, 0.94);
    color: #222222;
    z-index: 9000;
    font-size: 14px;
    border: 1px solid #555555;
    border-radius: 8px;
    padding: 12px;
    box-shadow: 0 0 15px rgba(0, 0, 0, 0.25);
">
    <b style="font-size: 16px;">
        Syria
    </b>
    <br>

    <span style="
        color: #226c4c;
        font-weight: bold;
    ">
        Population Raster Aggregation
        (Approximately 1 km)
    </span>

    <small style="
        display: block;
        margin-top: 7px;
        line-height: 1.35;
        color: #333333;
    ">
        The WorldPop 2026 population raster was aggregated
        from approximately 100 m source cells to
        approximately 1 km output cells.
        Sum resampling was used because source values
        represent estimated population per grid cell.
        Display colours use logarithmic scaling and are
        capped at the 99th percentile.
    </small>

    <div style="
        margin-top: 10px;
        padding-top: 7px;
        font-size: 11px;
        line-height: 1.35;
        color: #555555;
        border-top: 1px solid #aaaaaa;
    ">
        Source:
        <a
            href="https://www.worldpop.org/"
            target="_blank"
            style="
                color: #226c4c;
                text-decoration: none;
                font-weight: bold;
            "
        >
            WorldPop 2026
        </a><br>

        Source dimensions:
        <b>{source_width:,} × {source_height:,}</b><br>

        Output dimensions:
        <b>{saved_width:,} × {saved_height:,}</b><br>

        Source estimated population:
        <b>{source_population_total:,.0f}</b><br>

        Aggregated estimated population:
        <b>{saved_population_total:,.0f}</b><br>

        Population preservation:
        <b>{population_preservation_percent:.6f}%</b><br>

        Method:
        Sum Resampling / GeoTIFF Export /
        Logarithmic Display
    </div>
</div>
"""

m.get_root().html.add_child(Element(information_panel_html))

In [ ]:
# 21
# Add the population legend and layer control
# 人口凡例とレイヤー切替コントロールを追加する

legend_labels = [f"{value:,.0f}" for value in legend_values]

legend_html = f"""
<div style="
    position: fixed;
    right: 35px;
    bottom: 25px;
    width: 350px;
    background-color: rgba(255, 255, 255, 0.94);
    color: #222222;
    z-index: 9000;
    font-size: 12px;
    border: 1px solid #555555;
    border-radius: 8px;
    padding: 12px;
    box-shadow: 0 0 12px rgba(0, 0, 0, 0.22);
">
    <b style="font-size: 14px;">
        Estimated population per aggregated grid cell
    </b>

    <div style="
        width: 100%;
        height: 14px;
        margin-top: 10px;
        background: linear-gradient(
            to right,
            rgba(123, 181, 160, 0),
            #7bb5a0 20%,
            #2e9166 40%,
            #226c4c 60%,
            #184d36 80%,
            #0f3122 100%
        );
        border: 1px solid #777777;
    "></div>

    <div style="
        display: flex;
        justify-content: space-between;
        margin-top: 3px;
        font-size: 10px;
    ">
        <span>{legend_labels[0]}</span>
        <span>{legend_labels[1]}</span>
        <span>{legend_labels[2]}</span>
        <span>{legend_labels[3]}</span>
        <span>{legend_labels[4]}</span>
    </div>

    <div style="
        margin-top: 10px;
        padding-top: 7px;
        border-top: 1px solid #aaaaaa;
        font-size: 11px;
        line-height: 1.4;
        color: #555555;
    ">
        Display scaling: logarithmic<br>
        Display cap: 99th percentile<br>
        Output resolution: approximately 1 km<br>
        CRS: EPSG:4326
    </div>
</div>
"""

m.get_root().html.add_child(Element(legend_html))

folium.LayerControl(
    position="topright",
    collapsed=False,
).add_to(m)

In [ ]:
# 22
# Save and display the interactive map
# インタラクティブ地図を保存し、Notebook上に表示する

m.save(output_path)

print(f"Map saved to: {output_path}")

m